# Retrieval Flow + Retrieval Evaluation

Evaluates minsearch keyword retrieval on the ground-truth question set
with **Hit Rate** and **MRR**, then tunes per-field boost weights via
random search on a validation split and reports before/after numbers on
a held-out test split.

Results from the committed dataset (k=5, 100 validation / 195 test):

| Configuration | Hit Rate (test) | MRR (test) |
|---|---|---|
| Baseline (no boosts) | 73.3% | 0.526 |
| Tuned boosts | **76.4%** | **0.561** |

Tuned weights: `section=0.66, district=1.52, category=0.08, title=0.60, text=1.95`

In [ ]:
import sys
sys.path.append("..")

import random

import pandas as pd

from zoning_assistant.minsearch import Index

In [ ]:
df = pd.read_csv("../data/zoning.csv")
documents = df.to_dict(orient="records")

index = Index(
    text_fields=["section", "district", "category", "title", "text"],
    keyword_fields=["id"],
)
index.fit(documents)

## Ground truth and validation/test split

In [ ]:
gt = pd.read_csv("../data/ground-truth-retrieval.csv").to_dict(orient="records")

random.seed(0)
random.shuffle(gt)

gt_val = gt[:100]
gt_test = gt[100:]
len(gt_val), len(gt_test)

## Metrics

- **Hit Rate**: fraction of questions where the correct record appears anywhere in the top-k.
- **MRR**: average of 1/rank of the correct record (0 if absent) — rewards ranking it high, not just retrieving it.

In [ ]:
def hit_rate(relevance_total):
    return sum(True in line for line in relevance_total) / len(relevance_total)


def mrr(relevance_total):
    total = 0.0
    for line in relevance_total:
        for rank, rel in enumerate(line):
            if rel:
                total += 1 / (rank + 1)
                break
    return total / len(relevance_total)


def evaluate(ground_truth, search_function):
    relevance_total = []
    for q in ground_truth:
        results = search_function(q)
        relevance_total.append([d["id"] == q["id"] for d in results])
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

## Baseline: no boosts, k=5

In [ ]:
def minsearch_search(query, boost=None, num_results=5):
    return index.search(
        query=query,
        filter_dict={},
        boost_dict=boost or {},
        num_results=num_results,
    )


baseline_val = evaluate(gt_val, lambda q: minsearch_search(q["question"]))
baseline_test = evaluate(gt_test, lambda q: minsearch_search(q["question"]))
baseline_val, baseline_test

## Tuning boost weights (random search on the validation split)

In [ ]:
def simple_optimize(param_ranges, objective_function, n_iterations=30):
    best_params, best_score = None, float("-inf")
    for _ in range(n_iterations):
        current = {f: random.uniform(lo, hi) for f, (lo, hi) in param_ranges.items()}
        score = objective_function(current)
        if score > best_score:
            best_score, best_params = score, current
    return best_params


param_ranges = {
    "section": (0.0, 3.0),
    "district": (0.0, 3.0),
    "category": (0.0, 3.0),
    "title": (0.0, 3.0),
    "text": (0.0, 3.0),
}


def objective(boost_params):
    return evaluate(
        gt_val, lambda q: minsearch_search(q["question"], boost=boost_params)
    )["mrr"]


random.seed(42)
best_params = simple_optimize(param_ranges, objective, n_iterations=30)
{k: round(v, 2) for k, v in best_params.items()}

## Before/after on the held-out test split

In [ ]:
tuned_test = evaluate(
    gt_test, lambda q: minsearch_search(q["question"], boost=best_params)
)
print("baseline test:", baseline_test)
print("tuned test:   ", tuned_test)

Keyword search misses synonyms. Checking hit rate on ADU questions
phrased differently ("granny flat", "backyard cottage", ...) vs the
overall rate.

In [ ]:
SYNONYMS = ["granny flat", "backyard cottage", "guest house", "secondary apartment"]

gt_df = pd.read_csv("../data/ground-truth-retrieval.csv")
syn_rows = gt_df[gt_df.question.str.contains("|".join(SYNONYMS), case=False)]


def hits(rows):
    ok = 0
    for r in rows.itertuples():
        res = minsearch_search(r.question, boost=best_params)
        ok += any(d["id"] == r.id for d in res)
    return ok / len(rows)


print(f"synonym-phrased ADU questions ({len(syn_rows)}): hit rate {hits(syn_rows):.0%}")
print(f"overall test hit rate: {tuned_test['hit_rate']:.0%}")

**Observed**: synonym-phrased ADU questions hit ~41% vs ~76% overall.
This gap is why Project 2 adds pgvector/semantic search.